# DINOv2 cosine distance -> training features (Colab)

Adds a `dinov2_dist` column to `combined_training_data.csv`.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. On CPU this takes ~10x longer.

### What the number is

Embeddings are L2-normalised, so cosine similarity is a plain dot product:

```
dinov2_dist = 1 - cos_sim        range [0, 2], 0 = identical
```

Same orientation as the 6 hash columns (0 = identical, larger = more different), which is
what lets it drop into the existing feature vector unchanged.

**Not** the same scale as the `ann_distance` FAISS reports — that is *squared* L2 over unit
vectors, i.e. `2*(1 - cos_sim)`, range [0, 4]. Do not mix the two.

### Model parity

Preprocessing here (open -> RGB -> resize 224 -> processor -> CLS token -> L2 normalise)
mirrors `hashing/retriever/embedder.py` step for step, so the training feature matches the
feature the live retriever serves. Keep `MODEL_SIZE` equal to the retriever's `MODEL_SIZE`.

Batching changes only how many images ride through the model at once, not the arithmetic
per image — every input is resized to 224x224 first, so there is no padding variance.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
print(torch.cuda.is_available())        # True = có GPU
print(torch.cuda.get_device_name(0))

True
Tesla T4


## 2. Locate the inputs

The CSV mixes two datasets whose paths are relative to **different roots**:

| Path prefix in CSV | Lives under |
|---|---|
| `references/`, `queries/` | the DISC21 `filtered_images` folder |
| `images/`, `images_variants/` | the repo's `hashing/data/own` folder |

One bounded walk of Drive finds all three inputs. If it picks the wrong candidate, set the
variables by hand at the top and re-run this cell.

In [3]:
import os
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive')

# ---- Set manually to override auto-discovery -----------------------------
DISC21_ROOT = None   # dir holding queries/ and references/
OWN_ROOT    = None   # dir holding images/ and images_variants/
INPUT_CSV   = None   # combined_training_data.csv
OUT_DIR     = None   # defaults to the CSV's folder
# --------------------------------------------------------------------------

MAX_DEPTH = 7   # Drive FUSE is slow; cap the walk rather than scanning everything

def scan(root, max_depth=MAX_DEPTH):
    disc21, own, csvs = [], [], []
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        p = Path(dirpath)
        if len(p.relative_to(root).parts) >= max_depth:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith('.')]
        if (p / 'queries').is_dir() and (p / 'references').is_dir():
            disc21.append(p)
        if (p / 'images').is_dir() and (p / 'images_variants').is_dir():
            own.append(p)
        if 'combined_training_data.csv' in filenames:
            csvs.append(p / 'combined_training_data.csv')
    return disc21, own, csvs

if DISC21_ROOT is None or OWN_ROOT is None or INPUT_CSV is None:
    print('Scanning Drive (this takes a minute)...')
    d_hits, o_hits, c_hits = scan(DRIVE)
    DISC21_ROOT = DISC21_ROOT or (d_hits[0] if d_hits else None)
    OWN_ROOT    = OWN_ROOT    or (o_hits[0] if o_hits else None)
    INPUT_CSV   = INPUT_CSV   or (c_hits[0] if c_hits else None)
    for label, hits in (('DISC21', d_hits), ('OWN', o_hits), ('CSV', c_hits)):
        if len(hits) > 1:
            print(f'  NOTE: {len(hits)} {label} candidates, using the first:')
            for h in hits[:5]:
                print('        ', h)

OUT_DIR = Path(OUT_DIR) if OUT_DIR else (Path(INPUT_CSV).parent if INPUT_CSV else DRIVE / 'dam_dinov2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print()
print('DISC21_ROOT :', DISC21_ROOT or 'NOT FOUND')
print('OWN_ROOT    :', OWN_ROOT    or 'NOT FOUND')
print('INPUT_CSV   :', INPUT_CSV   or 'NOT FOUND')
print('OUT_DIR     :', OUT_DIR)

if INPUT_CSV is None:
    raise SystemExit('Upload combined_training_data.csv to Drive, or set INPUT_CSV above.')
if DISC21_ROOT is None and OWN_ROOT is None:
    raise SystemExit('Neither image root found — nothing to embed.')
if OWN_ROOT is None:
    print('\nWARNING: own-dataset images not on Drive. Those rows will be left BLANK.\n'
          '         Upload hashing/data/own to Drive to fill them.')
if DISC21_ROOT is None:
    print('\nWARNING: DISC21 images not on Drive. Those rows will be left BLANK.')

Scanning Drive (this takes a minute)...
  NOTE: 2 OWN candidates, using the first:
         /content/drive/MyDrive/inpaint_project/Dataset/own
         /content/drive/MyDrive/inpaint_project/own

DISC21_ROOT : /content/drive/MyDrive/inpaint_project/Dataset/disc21/filtered_images
OWN_ROOT    : /content/drive/MyDrive/inpaint_project/Dataset/own
INPUT_CSV   : /content/drive/MyDrive/inpaint_project/combined_training_data.csv
OUT_DIR     : /content/drive/MyDrive/inpaint_project


## 3. Dependencies

Colab ships both. The install line is only for a stale runtime.

In [4]:
# !pip -q install -U transformers

import torch, transformers

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('device      :', device)
if device.type == 'cpu':
    print('\nNo GPU. Runtime -> Change runtime type -> T4 GPU, then re-run from cell 1.')

torch       : 2.11.0+cu128
transformers: 5.13.1
device      : cuda


## 4. Load the CSV and resolve every image path

In [5]:
import csv

OWN_PREFIXES    = ('images/', 'images_variants/')
DISC21_PREFIXES = ('references/', 'queries/')

FEATURE_COL = 'dinov2_dist'
HASH_COLS = ['ahash_dist', 'phash_dist', 'dhash_dist',
             'hsvhash_dist', 'rhash_dist', 'chash_dist']

with open(INPUT_CSV, newline='') as f:
    reader = csv.DictReader(f)
    fieldnames = list(reader.fieldnames or [])
    rows = list(reader)
print('rows:', len(rows))

def resolve(rel):
    rel = rel.strip().replace('\\', '/')
    if rel.startswith(OWN_PREFIXES):
        return (Path(OWN_ROOT) / rel) if OWN_ROOT else None
    if rel.startswith(DISC21_PREFIXES):
        return (Path(DISC21_ROOT) / rel) if DISC21_ROOT else None
    return None

resolved, missing = {}, {}
for r in rows:
    for key in ('image_a', 'image_b'):
        rel = r[key]
        if rel in resolved or rel in missing:
            continue
        p = resolve(rel)
        if p is not None and p.exists():
            resolved[rel] = p
        else:
            missing[rel] = p

print(f'unique images : {len(resolved) + len(missing)}')
print(f'  resolved    : {len(resolved)}')
print(f'  missing     : {len(missing)}')
if missing:
    print('\nfirst few missing:')
    for rel in list(missing)[:5]:
        print('  ', rel, '->', missing[rel])

rows: 2223
unique images : 2115
  resolved    : 2115
  missing     : 0


## 5. Load DINOv2

`MODEL_SIZE` must match the retriever's `MODEL_SIZE` env var (`small` by default in
`hashing/main.py`), or the training feature and the served feature come from different models.

In [6]:
from transformers import AutoImageProcessor, AutoModel

MODEL_SIZE = 'small'   # small | base | large | giant

name = f'facebook/dinov2-{MODEL_SIZE}'
print('loading', name)
proc  = AutoImageProcessor.from_pretrained(name)
model = AutoModel.from_pretrained(name).eval().to(device)
print('loaded on', device)

loading facebook/dinov2-small


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

loaded on cuda


## 6. Embed every unique image

Each unique image is embedded **once** — the own-dataset rows reuse ~57 images across ~165
pairs, so pair count is a bad proxy for work.

Vectors are checkpointed to Drive every `SAVE_EVERY` batches. If Colab disconnects, re-run
this cell and it resumes from the cache instead of starting over.

In [7]:
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

BATCH      = 32
SAVE_EVERY = 20   # batches between checkpoints
CACHE_PATH = OUT_DIR / f'dinov2_{MODEL_SIZE}_embeddings.npz'

cache = {}
if CACHE_PATH.exists():
    z = np.load(CACHE_PATH, allow_pickle=False)
    cache = {k: v for k, v in zip(z['keys'].tolist(), z['vecs'])}
    print(f'resumed {len(cache)} embeddings from {CACHE_PATH}')

def save_cache():
    if cache:
        np.savez(CACHE_PATH,
                 keys=np.array(list(cache.keys())),
                 vecs=np.stack(list(cache.values())))

def embed_batch(paths):
    imgs, kept = [], []
    for sp in paths:
        try:
            # Mirrors ImageEmbedder.get_embedding_from_path exactly.
            imgs.append(Image.open(sp).convert('RGB').resize((224, 224)))
            kept.append(sp)
        except Exception as e:
            print('skip', sp, e)
    if not imgs:
        return
    inputs = proc(images=imgs, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        h = model(**inputs).last_hidden_state[:, 0, :]
        h = h / h.norm(dim=-1, keepdim=True)
    for sp, v in zip(kept, h.cpu().numpy()):
        cache[sp] = v

todo = sorted({str(p) for p in resolved.values()} - set(cache))
print('to embed:', len(todo))

for i in tqdm(range(0, len(todo), BATCH)):
    embed_batch(todo[i:i + BATCH])
    if (i // BATCH) % SAVE_EVERY == 0:
        save_cache()
save_cache()

print('cached embeddings:', len(cache))
print('checkpoint       :', CACHE_PATH)

to embed: 2115


  0%|          | 0/67 [00:00<?, ?it/s]

cached embeddings: 2115
checkpoint       : /content/drive/MyDrive/inpaint_project/dinov2_small_embeddings.npz


## 7. Cosine distance per pair

In [8]:
skipped = 0
for r in rows:
    pa, pb = resolved.get(r['image_a']), resolved.get(r['image_b'])
    va = cache.get(str(pa)) if pa else None
    vb = cache.get(str(pb)) if pb else None
    if va is None or vb is None:
        r[FEATURE_COL] = ''          # left blank, never silently zero
        skipped += 1
        continue
    # Unit-norm vectors, so the dot product IS the cosine similarity.
    # Clip absorbs float drift that would push |cos| just past 1.
    cos = float(np.clip(np.dot(va, vb), -1.0, 1.0))
    r[FEATURE_COL] = f'{1.0 - cos:.6f}'

print(f'filled {len(rows) - skipped}/{len(rows)} rows')
if skipped:
    print(f'{skipped} row(s) left BLANK — drop or impute before training')

filled 2223/2223 rows


## 8. Save back to Drive

In [9]:
if FEATURE_COL not in fieldnames:
    fieldnames.append(FEATURE_COL)

OUTPUT_CSV = OUT_DIR / 'combined_training_data_with_dinov2.csv'
with open(OUTPUT_CSV, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    w.writeheader()
    w.writerows(rows)

print('saved', len(rows), 'rows ->', OUTPUT_CSV)

saved 2223 rows -> /content/drive/MyDrive/inpaint_project/combined_training_data_with_dinov2.csv


## 9. Does it actually help?

The six hashes collapse on DISC21 — positives there average ~0.45 normalised Hamming on
ahash and ~0.51 on phash, barely off a coin flip, because DISC21 edits are semantic rather
than the photometric transforms the hashes were tuned for. That subset is broken out
separately below, because an aggregate gap can look healthy while the hard subset sits at
chance.

In [ ]:
def col(rs, name):
    out = []
    for r in rs:
        v = r.get(name, '')
        if v != '':
            try:
                out.append(float(v))
            except ValueError:
                pass
    return np.array(out)

def table(rs, title):
    pos = [r for r in rs if str(r.get('label', '')).strip() == '1']
    neg = [r for r in rs if str(r.get('label', '')).strip() == '0']
    if not pos or not neg:
        return
    print(f'\n{title}  ({len(pos)} pos / {len(neg)} neg)')
    print(f"{'feature':<14}{'pos':>10}{'neg':>10}{'gap':>10}")
    print('-' * 44)
    for name in HASH_COLS + [FEATURE_COL]:
        p, n = col(pos, name), col(neg, name)
        if p.size == 0 or n.size == 0:
            continue
        mark = '  <-- new' if name == FEATURE_COL else ''
        print(f'{name:<14}{p.mean():>10.4f}{n.mean():>10.4f}{n.mean() - p.mean():>10.4f}{mark}')

def is_disc(r):
    return str(r.get('transformation', '')).startswith('disc21')

table(rows, 'ALL PAIRS')
table([r for r in rows if is_disc(r)], 'DISC21 ONLY (the hard subset)')
table([r for r in rows if not is_disc(r)], 'OWN DATASET ONLY')
print('\ngap = negative mean - positive mean. Larger is better separation.')

## Next

The deployed logreg takes **6** features — `FEATURES_ORDER` in `hashing/api/similarity.py`.
Adding this column to the CSV does not change what runs in production, and feeding the
current model 7 features will raise. To adopt it:

1. Retrain on the 7-feature CSV (`copymint_logreg_baseline_ver1.ipynb`).
2. Add `dinov2_dist` to `FEATURES_ORDER`.
3. Compute the same distance at serve time — the retriever already embeds the query image
   for the FAISS recall stage, so the vector is in hand; it just is not being fed to the
   combiner today.
4. Bump the model version. `verifications.model_version` in the backend exists so a past
   verdict can be traced to the model that produced it — a retrain that reuses the old
   version string breaks that.